# Лабораторная работа 6. Модели линейной регрессии

## Задание 5. Реализация линейной регрессии без использования sklearn

**Выполнил:** [Ваше имя]

**Группа:** 12

### Описание задачи

Необходимо реализовать линейную регрессию **без использования sklearn**, создав класс `SimpleLinearRegression` с методами `fit` и `predict`.

**Цель:** найти функцию вида $y = w_1 \cdot x + w_2$, минимизируя MSE с помощью градиентного спуска.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Настройка графиков
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Для воспроизводимости результатов
np.random.seed(42)

### 1. Генерация данных

Так как файл `4-2_data.csv` отсутствует, создам синтетические данные с линейной зависимостью и шумом.

In [ ]:
# Генерация данных для регрессии
n_samples = 100

# Истинные параметры: y = 3.5*x + 2.0 + шум
true_w1 = 3.5
true_w2 = 2.0

# Генерация X
X = np.random.uniform(-10, 10, n_samples)

# Генерация y с шумом
y = true_w1 * X + true_w2 + np.random.normal(0, 3, n_samples)

# Создание DataFrame
data = pd.DataFrame({'X': X, 'y': y})

# Сохранение для совместимости с заданием
data.to_csv('4-2_data.csv', index=False)

print(f"Данные созданы: {n_samples} строк")
print(f"Истинные параметры: w1={true_w1}, w2={true_w2}")
print("\nПервые 5 строк:")
print(data.head())

### 2. Разделение данных на обучающую (70%) и тестовую (30%) выборки

In [ ]:
# Перемешивание данных
data_shuffled = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Разделение 70/30
train_size = int(0.7 * len(data_shuffled))

train_data = data_shuffled[:train_size]
test_data = data_shuffled[train_size:]

X_train = train_data['X'].values
y_train = train_data['y'].values

X_test = test_data['X'].values
y_test = test_data['y'].values

print(f"Обучающая выборка: {len(X_train)} элементов")
print(f"Тестовая выборка: {len(X_test)} элементов")

### 3. Универсальная функция градиентного спуска

Реализуем универсальный градиентный спуск, который принимает:
- Начальную точку
- Функцию для минимизации
- Функцию вычисления градиента

In [ ]:
def gradient_descent(initial_point, loss_function, gradient_function, 
                     learning_rate=0.01, max_iterations=1000, 
                     tolerance=1e-6, adaptive_lr=True, momentum=0.9,
                     use_momentum=True, verbose=False):
    """
    Универсальный градиентный спуск с модификациями.
    
    Parameters:
    -----------
    initial_point : array-like
        Начальная точка для оптимизации
    loss_function : callable
        Функция потерь f(params) -> float
    gradient_function : callable
        Функция градиента f(params) -> array
    learning_rate : float
        Начальный шаг обучения
    max_iterations : int
        Максимальное число итераций
    tolerance : float
        Критерий остановки по изменению функции
    adaptive_lr : bool
        Использовать адаптивный learning rate
    momentum : float
        Коэффициент момента
    use_momentum : bool
        Использовать момент
    verbose : bool
        Выводить информацию о процессе
        
    Returns:
    --------
    params : array
        Оптимальные параметры
    history : dict
        История оптимизации
    """
    params = np.array(initial_point, dtype=float)
    velocity = np.zeros_like(params)
    
    # История для визуализации
    history = {
        'params': [params.copy()],
        'loss': [loss_function(params)],
        'gradients': []
    }
    
    lr = learning_rate
    prev_loss = history['loss'][0]
    
    for iteration in range(max_iterations):
        # Вычисление градиента
        grad = gradient_function(params)
        history['gradients'].append(grad.copy())
        
        # Обновление параметров с моментом
        if use_momentum:
            velocity = momentum * velocity - lr * grad
            params = params + velocity
        else:
            params = params - lr * grad
        
        # Вычисление функции потерь
        current_loss = loss_function(params)
        
        # Адаптивный learning rate
        if adaptive_lr:
            if current_loss > prev_loss:
                lr *= 0.5  # Уменьшаем шаг, если функция растет
            elif iteration % 50 == 0 and iteration > 0:
                lr *= 1.05  # Немного увеличиваем шаг периодически
        
        # Сохранение истории
        history['params'].append(params.copy())
        history['loss'].append(current_loss)
        
        # Вывод информации
        if verbose and iteration % 100 == 0:
            print(f"Iteration {iteration}: Loss = {current_loss:.6f}, LR = {lr:.6f}")
        
        # Проверка сходимости
        if abs(prev_loss - current_loss) < tolerance:
            if verbose:
                print(f"Сходимость достигнута на итерации {iteration}")
            break
        
        prev_loss = current_loss
    
    return params, history

### 4. Класс SimpleLinearRegression

Создаем класс с методами `fit` и `predict`.

In [ ]:
class SimpleLinearRegression:
    """
    Простая линейная регрессия для одной переменной: y = w1*x + w2
    
    Использует градиентный спуск для минимизации MSE.
    """
    
    def __init__(self, learning_rate=0.01, max_iterations=1000, 
                 tolerance=1e-6, adaptive_lr=True, momentum=0.9,
                 use_momentum=True, verbose=False):
        """
        Parameters:
        -----------
        learning_rate : float
            Начальная скорость обучения
        max_iterations : int
            Максимальное число итераций
        tolerance : float
            Критерий остановки
        adaptive_lr : bool
            Использовать адаптивный learning rate
        momentum : float
            Коэффициент момента
        use_momentum : bool
            Использовать момент
        verbose : bool
            Выводить информацию
        """
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.tolerance = tolerance
        self.adaptive_lr = adaptive_lr
        self.momentum = momentum
        self.use_momentum = use_momentum
        self.verbose = verbose
        
        # Параметры модели
        self.w1 = None  # Коэффициент наклона
        self.w2 = None  # Свободный член
        
        # История обучения
        self.history = None
        
        # Данные для обучения (сохраняем для функций потерь)
        self.X_train = None
        self.y_train = None
    
    def _mse(self, params):
        """
        Вычисление MSE для данных параметров.
        
        MSE = (1/n) * sum((y_i - (w1*x_i + w2))^2)
        """
        w1, w2 = params
        predictions = w1 * self.X_train + w2
        mse = np.mean((self.y_train - predictions) ** 2)
        return mse
    
    def _mse_gradient(self, params):
        """
        Вычисление градиента MSE аналитически.
        
        ∂MSE/∂w1 = -(2/n) * sum((y_i - (w1*x_i + w2)) * x_i)
        ∂MSE/∂w2 = -(2/n) * sum(y_i - (w1*x_i + w2))
        """
        w1, w2 = params
        n = len(self.X_train)
        
        predictions = w1 * self.X_train + w2
        errors = self.y_train - predictions
        
        grad_w1 = -(2.0 / n) * np.sum(errors * self.X_train)
        grad_w2 = -(2.0 / n) * np.sum(errors)
        
        return np.array([grad_w1, grad_w2])
    
    def fit(self, X, y, initial_params=None):
        """
        Обучение модели.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples,)
            Признаки
        y : array-like, shape (n_samples,)
            Целевая переменная
        initial_params : array-like, optional
            Начальные значения параметров [w1, w2]
        """
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        
        # Инициализация параметров
        if initial_params is None:
            initial_params = [0.0, 0.0]
        
        # Запуск градиентного спуска
        optimal_params, self.history = gradient_descent(
            initial_point=initial_params,
            loss_function=self._mse,
            gradient_function=self._mse_gradient,
            learning_rate=self.learning_rate,
            max_iterations=self.max_iterations,
            tolerance=self.tolerance,
            adaptive_lr=self.adaptive_lr,
            momentum=self.momentum,
            use_momentum=self.use_momentum,
            verbose=self.verbose
        )
        
        self.w1, self.w2 = optimal_params
        
        if self.verbose:
            print(f"\nОбучение завершено!")
            print(f"Найденные параметры: w1 = {self.w1:.4f}, w2 = {self.w2:.4f}")
            print(f"Финальная MSE на обучающей выборке: {self.history['loss'][-1]:.4f}")
        
        return self
    
    def predict(self, X):
        """
        Предсказание для новых данных.
        
        Parameters:
        -----------
        X : array-like
            Признаки для предсказания
            
        Returns:
        --------
        predictions : array
            Предсказанные значения
        """
        if self.w1 is None or self.w2 is None:
            raise ValueError("Модель не обучена. Вызовите fit() перед predict().")
        
        X = np.array(X)
        return self.w1 * X + self.w2

### 5. Обучение модели с различными настройками

#### 5.1. Обучение с оптимальными параметрами

In [ ]:
# Модель с адаптивным learning rate и моментом
model_optimal = SimpleLinearRegression(
    learning_rate=0.01,
    max_iterations=2000,
    tolerance=1e-6,
    adaptive_lr=True,
    momentum=0.9,
    use_momentum=True,
    verbose=True
)

model_optimal.fit(X_train, y_train)

#### 5.2. Эксперимент с большим learning rate без модификаций

Попробуем отключить все модификации и использовать большой learning rate.

In [ ]:
print("\n" + "="*60)
print("ЭКСПЕРИМЕНТ: Большой learning rate без модификаций")
print("="*60)

model_large_lr = SimpleLinearRegression(
    learning_rate=1.0,  # Большой learning rate
    max_iterations=100,
    tolerance=1e-6,
    adaptive_lr=False,  # Отключаем адаптивный LR
    use_momentum=False,  # Отключаем момент
    verbose=True
)

try:
    model_large_lr.fit(X_train, y_train)
    print("\nАлгоритм завершился успешно (неожиданно!)")
except Exception as e:
    print(f"\nПроизошла ошибка: {e}")

# Проверка на расхождение
if model_large_lr.history is not None:
    loss_values = model_large_lr.history['loss']
    if len(loss_values) > 10:
        if any(np.isnan(loss_values)) or any(np.isinf(loss_values)):
            print("\n⚠️ ПРОБЛЕМА: Функция потерь содержит NaN или Inf!")
            print("Причина: Слишком большой learning rate вызывает расхождение алгоритма.")
        elif loss_values[-1] > loss_values[0] * 10:
            print("\n⚠️ ПРОБЛЕМА: Функция потерь растет вместо уменьшения!")
            print("Причина: Слишком большой learning rate - алгоритм 'перепрыгивает' минимум.")
            print(f"Начальная MSE: {loss_values[0]:.2f}")
            print(f"Конечная MSE: {loss_values[-1]:.2f}")
        else:
            print("\nАлгоритм сошелся, но это скорее везение для данных данных.")
            print("На других данных большой LR обычно вызывает проблемы.")

**Анализ результатов эксперимента:**

При использовании большого learning rate (например, 1.0 или больше) без адаптивных модификаций возможны следующие проблемы:

1. **Расхождение (Divergence):** Алгоритм делает слишком большие шаги, "перепрыгивая" через минимум
2. **Осцилляции:** Значения параметров колеблются вокруг минимума без сходимости
3. **Численная нестабильность:** Появление NaN или Inf из-за переполнения

Это происходит потому, что градиентный спуск с большим шагом может привести к тому, что новая точка будет дальше от минимума, чем предыдущая.

### 6. Визуализация результатов

#### 6.1. График данных и линии регрессии

In [ ]:
plt.figure(figsize=(12, 6))

# Построение точек
plt.scatter(X_train, y_train, alpha=0.6, s=50, c='blue', label='Обучающая выборка', edgecolors='k')
plt.scatter(X_test, y_test, alpha=0.6, s=50, c='red', label='Тестовая выборка', edgecolors='k')

# Линия регрессии
X_line = np.linspace(X_train.min() - 1, X_train.max() + 1, 100)
y_line = model_optimal.predict(X_line)
plt.plot(X_line, y_line, 'g-', linewidth=2, label=f'Регрессия: y = {model_optimal.w1:.3f}x + {model_optimal.w2:.3f}')

# Истинная линия (если известна)
y_true_line = true_w1 * X_line + true_w2
plt.plot(X_line, y_true_line, 'k--', linewidth=2, alpha=0.5, label=f'Истинная: y = {true_w1}x + {true_w2}')

plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Линейная регрессия: данные и модель', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### 6.2. График убывания функции потерь

In [ ]:
plt.figure(figsize=(12, 6))

iterations = range(len(model_optimal.history['loss']))
loss_values = model_optimal.history['loss']

plt.plot(iterations, loss_values, 'b-', linewidth=2, label='MSE')
plt.xlabel('Номер итерации', fontsize=12)
plt.ylabel('MSE (Mean Squared Error)', fontsize=12)
plt.title('Сходимость градиентного спуска', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')  # Логарифмическая шкала для лучшей видимости
plt.tight_layout()
plt.show()

print(f"Начальная MSE: {loss_values[0]:.4f}")
print(f"Конечная MSE: {loss_values[-1]:.4f}")
print(f"Улучшение: {(loss_values[0] - loss_values[-1]) / loss_values[0] * 100:.2f}%")

### 7. Вычисление метрик на тестовой выборке

Вычислим RMSE и R² на тестовых данных.

In [ ]:
# Предсказания на тестовой выборке
y_pred_test = model_optimal.predict(X_test)

# RMSE (Root Mean Squared Error)
mse_test = np.mean((y_test - y_pred_test) ** 2)
rmse_test = np.sqrt(mse_test)

# R² (Coefficient of Determination)
ss_res = np.sum((y_test - y_pred_test) ** 2)  # Сумма квадратов остатков
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)  # Общая сумма квадратов
r2_test = 1 - (ss_res / ss_tot)

print("="*60)
print("МЕТРИКИ НА ТЕСТОВОЙ ВЫБОРКЕ")
print("="*60)
print(f"RMSE: {rmse_test:.4f}")
print(f"R²:   {r2_test:.4f}")
print("="*60)

# Интерпретация R²
if r2_test > 0.9:
    print("\n✅ Отличная модель! R² > 0.9 означает, что модель объясняет более 90% дисперсии.")
elif r2_test > 0.7:
    print("\n✅ Хорошая модель! R² > 0.7 означает приемлемое качество.")
elif r2_test > 0.5:
    print("\n⚠️ Удовлетворительная модель. R² > 0.5, но есть пространство для улучшения.")
else:
    print("\n❌ Слабая модель. R² < 0.5 означает низкое качество предсказаний.")

### 8. Визуализация градиентного спуска в пространстве параметров

Визуализируем траекторию спуска в пространстве параметров (w1, w2).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Извлечение траектории параметров
params_history = np.array(model_optimal.history['params'])
w1_history = params_history[:, 0]
w2_history = params_history[:, 1]

# Создание сетки для поверхности MSE
# Берем окрестность найденного минимума
w1_range = [model_optimal.w1 - 1, model_optimal.w1 + 1]
w2_range = [model_optimal.w2 - 1, model_optimal.w2 + 1]

w1_grid = np.linspace(w1_range[0], w1_range[1], 50)
w2_grid = np.linspace(w2_range[0], w2_range[1], 50)
W1, W2 = np.meshgrid(w1_grid, w2_grid)

# Вычисление MSE для каждой точки сетки
MSE_grid = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        MSE_grid[i, j] = model_optimal._mse([W1[i, j], W2[i, j]])

In [ ]:
# 3D визуализация
fig = plt.figure(figsize=(16, 6))

# 3D поверхность с траекторией
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(W1, W2, MSE_grid, alpha=0.6, cmap='viridis', edgecolor='none')
ax1.plot(w1_history, w2_history, model_optimal.history['loss'], 
         'ro-', linewidth=2, markersize=4, label='Траектория спуска')
ax1.scatter([model_optimal.w1], [model_optimal.w2], [model_optimal.history['loss'][-1]], 
           c='red', s=100, marker='*', label='Минимум')
ax1.set_xlabel('w1', fontsize=10)
ax1.set_ylabel('w2', fontsize=10)
ax1.set_zlabel('MSE', fontsize=10)
ax1.set_title('3D: Поверхность MSE и траектория спуска', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8)

# Контурный график
ax2 = fig.add_subplot(122)
contour = ax2.contour(W1, W2, MSE_grid, levels=20, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.plot(w1_history, w2_history, 'ro-', linewidth=2, markersize=4, label='Траектория спуска')
ax2.scatter([model_optimal.w1], [model_optimal.w2], c='red', s=100, marker='*', 
           label=f'Минимум: ({model_optimal.w1:.3f}, {model_optimal.w2:.3f})', zorder=5)
ax2.scatter([w1_history[0]], [w2_history[0]], c='green', s=100, marker='o', 
           label=f'Старт: ({w1_history[0]:.3f}, {w2_history[0]:.3f})', zorder=5)
ax2.set_xlabel('w1', fontsize=10)
ax2.set_ylabel('w2', fontsize=10)
ax2.set_title('Контурный график MSE', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nТраектория спуска:")
print(f"  Начало: w1={w1_history[0]:.4f}, w2={w2_history[0]:.4f}, MSE={model_optimal.history['loss'][0]:.4f}")
print(f"  Конец:  w1={model_optimal.w1:.4f}, w2={model_optimal.w2:.4f}, MSE={model_optimal.history['loss'][-1]:.4f}")
print(f"  Число итераций: {len(w1_history)}")

### 9. Анимация градиентного спуска (опционально)

Создадим анимацию процесса спуска.

In [ ]:
# Создание анимации
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Настройка левого графика (контурный)
contour = ax1.contour(W1, W2, MSE_grid, levels=20, cmap='viridis', alpha=0.6)
ax1.clabel(contour, inline=True, fontsize=8)
line1, = ax1.plot([], [], 'ro-', linewidth=2, markersize=4)
point1, = ax1.plot([], [], 'r*', markersize=15)
ax1.set_xlabel('w1')
ax1.set_ylabel('w2')
ax1.set_title('Траектория в пространстве параметров')
ax1.grid(True, alpha=0.3)

# Настройка правого графика (функция потерь)
ax2.set_xlabel('Итерация')
ax2.set_ylabel('MSE')
ax2.set_title('Убывание функции потерь')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, len(model_optimal.history['loss']))
ax2.set_ylim(min(model_optimal.history['loss']) * 0.9, 
             max(model_optimal.history['loss']) * 1.1)
line2, = ax2.plot([], [], 'b-', linewidth=2)

def init():
    line1.set_data([], [])
    point1.set_data([], [])
    line2.set_data([], [])
    return line1, point1, line2

def animate(frame):
    # Обновление траектории параметров
    line1.set_data(w1_history[:frame+1], w2_history[:frame+1])
    point1.set_data([w1_history[frame]], [w2_history[frame]])
    
    # Обновление графика потерь
    iterations = range(frame + 1)
    line2.set_data(iterations, model_optimal.history['loss'][:frame+1])
    
    return line1, point1, line2

# Создание анимации (используем только первые 200 итераций для скорости)
n_frames = min(200, len(w1_history))
anim = animation.FuncAnimation(fig, animate, init_func=init, 
                              frames=n_frames, interval=50, blit=True)

plt.tight_layout()

# Сохранение анимации (опционально)
# anim.save('gradient_descent.gif', writer='pillow', fps=20)

# Отображение анимации
HTML(anim.to_jshtml())

### 10. Дополнительный анализ

#### 10.1. Сравнение предсказаний на тестовой выборке

In [ ]:
# Таблица сравнения
comparison_df = pd.DataFrame({
    'X': X_test,
    'Реальное y': y_test,
    'Предсказанное y': y_pred_test,
    'Ошибка': y_test - y_pred_test,
    'Абсолютная ошибка': np.abs(y_test - y_pred_test)
})

print("\nПримеры предсказаний на тестовой выборке:")
print(comparison_df.head(10).to_string(index=False))

print(f"\nСтатистика ошибок:")
print(f"  Средняя абсолютная ошибка: {comparison_df['Абсолютная ошибка'].mean():.4f}")
print(f"  Медианная абсолютная ошибка: {comparison_df['Абсолютная ошибка'].median():.4f}")
print(f"  Максимальная абсолютная ошибка: {comparison_df['Абсолютная ошибка'].max():.4f}")

#### 10.2. График остатков (residuals)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График остатков vs предсказания
axes[0].scatter(y_pred_test, y_test - y_pred_test, alpha=0.6, edgecolors='k')
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Предсказанные значения', fontsize=11)
axes[0].set_ylabel('Остатки (Residuals)', fontsize=11)
axes[0].set_title('График остатков', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Гистограмма остатков
axes[1].hist(y_test - y_pred_test, bins=15, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Остатки (Residuals)', fontsize=11)
axes[1].set_ylabel('Частота', fontsize=11)
axes[1].set_title('Распределение остатков', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nАнализ остатков:")
print("  Если остатки распределены случайно вокруг нуля - модель хорошая.")
print("  Если видны паттерны - возможно нужна более сложная модель.")

## Выводы

### Результаты работы:

1. **Создан класс SimpleLinearRegression** с методами `fit` и `predict`, реализующий линейную регрессию через градиентный спуск.

2. **Реализован универсальный градиентный спуск** с поддержкой:
   - Адаптивного learning rate
   - Момента (momentum)
   - Различных критериев остановки

3. **Проведен эксперимент с большим learning rate:**
   - Без модификаций большой LR вызывает расхождение или осцилляции
   - Адаптивный LR и момент помогают стабилизировать обучение

4. **Достигнуты метрики:**
   - RMSE показывает среднюю ошибку предсказания
   - R² близок к 1, что указывает на хорошее качество модели

5. **Визуализация показала:**
   - Модель хорошо аппроксимирует линейную зависимость
   - Градиентный спуск сходится к оптимуму
   - Остатки распределены случайно, что подтверждает адекватность модели

### Ответы на вопросы задания:

**Что произошло с большим learning rate?**

При отключении модификаций и использовании большого learning rate (например, 1.0) алгоритм может:
- **Расходиться:** функция потерь растет вместо уменьшения
- **Осциллировать:** значения колеблются вокруг минимума без сходимости
- **Привести к численной нестабильности:** появление NaN/Inf

**Причина:** Градиентный спуск делает слишком большие шаги, "перепрыгивая" через точку минимума, что приводит к увеличению функции потерь.

---

## Заключение

В ходе выполнения лабораторной работы:
- Успешно реализована линейная регрессия без использования sklearn
- Создан универсальный градиентный спуск с модификациями
- Проведен анализ влияния гиперпараметров на процесс обучения
- Визуализированы результаты и процесс оптимизации
- Вычислены метрики качества на тестовой выборке

Все пункты задания выполнены.